<a href="https://colab.research.google.com/github/whorullost/TriggerESM/blob/main/TriggerESM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch --quiet

In [ ]:
!pip install biopython

In [ ]:
from transformers import AutoTokenizer, EsmModel
import torch

tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
model = EsmModel.from_pretrained("facebook/esm2_t6_8M_UR50D")

test_sequence = "MKTAYIAKQRQISFVKSHFSRQ"
inputs = tokenizer(test_sequence, return_tensors="pt")
outputs = model(**inputs)

print(outputs.last_hidden_state.shape)


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 31.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.Size([1, 24, 320])


In [ ]:
from io import StringIO
from Bio import SeqIO
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, EsmModel
import torch

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

rbp1_fasta = """>sp|P04050|RPB1_YEAST DNA-directed RNA polymerase II subunit RPB1 OS=Saccharomyces cerevisiae (strain ATCC 204508 / S288c) OX=559292 GN=RPO21 PE=1 SV=2
MVGQQYSSAPLRTVKEVQFGLFSPEEVRAISVAKIRFPETMDETQTRAKIGGLNDPRLGS
IDRNLKCQTCQEGMNECPGHFGHIDLAKPVFHVGFIAKIKKVCECVCMHCGKLLLDEHNE
LMRQALAIKDSKKRFAAIWTLCKTKMVCETDVPSEDDPTQLVSRGGCGNTQPTIRKDGLK
LVGSWKKDRATGDADEPELRVLSTEEILNIFKHISVKDFTSLGFNEVFSRPEWMILTCLP
VPPPPVRPSISFNESQRGEDDLTFKLADILKANISLETLEHNGAPHHAIEEAESLLQFHV
ATYMDNDIAGQPQALQKSGRPVKSIRARLKGKEGRIRGNLMGKRVDFSARTVISGDPNLE
LDQVGVPKSIAKTLTYPEVVTPYNIDRLTQLVRNGPNEHPGAKYVIRDSGDRIDLRYSKR
AGDIQLQYGWKVERHIMDNDPVLFNRQPSLHKMSMMAHRVKVIPYSTFRLNLSVTSPYNA
DFDGDEMNLHVPQSEETRAELSQLCAVPLQIVSPQSNKPCMGIVQDTLCGIRKLTLRDTF
IELDQVLNMLYWVPDWDGVIPTPAIIKPKPLWSGKQILSVAIPNGIHLQRFDEGTTLLSP
KDNGMLIIDGQIIFGVVEKKTVGSSNGGLIHVVTREKGPQVCAKLFGNIQKVVNFWLLHN
GFSTGIGDTIADGPTMREITETIAEAKKKVLDVTKEAQANLLTAKHGMTLRESFEDNVVR
FLNEARDKAGRLAEVNLKDLNNVKQMVMAGSKGSFINIAQMSACVGQQSVEGKRIAFGFV
DRTLPHFSKDDYSPESKGFVENSYLRGLTPQEFFFHAMGGREGLIDTAVKTAETGYIQRR
LVKALEDIMVHYDNTTRNSLGNVIQFIYGEDGMDAAHIEKQSLDTIGGSDAAFEKRYRVD
LLNTDHTLDPSLLESGSEILGDLKLQVLLDEEYKQLVKDRKFLREVFVDGEANWPLPVNI
RRIIQNAQQTFHIDHTKPSDLTIKDIVLGVKDLQENLLVLRGKNEIIQNAQRDAVTLFCC
LLRSRLATRRVLQEYRLTKQAFDWVLSNIEAQFLRSVVHPGEMVGVLAAQSIGEPATQMT
LNTFHFAGVASKKVTSGVPRLKEILNVAKNMKTPSLTVYLEPGHAADQEQAKLIRSAIEH
TTLKSVTIASEIYYDPDPRSTVIPEDEEIIQLHFSLLDEEAEQSFDQQSPWLLRLELDRA
AMNDKDLTMGQVGERIKQTFKNDLFVIWSEDNDEKLIIRCRVVRPKSLDAETEAEEDHML
KKIENTMLENITLRGVENIERVVMMKYDRKVPSPTGEYVKEPEWVLETDGVNLSEVMTVP
GIDPTRIYTNSFIDIMEVLGIEAGRAALYKEVYNVIASDGSYVNYRHMALLVDVMTTQGG
LTSVTRHGFNRSNTGALMRCSFEETVEILFEAGASAELDDCRGVSENVILGQMAPIGTGA
FDVMIDEESLVKYMPEQKITEIEDGQDGGVTPYSNESGLVNADLDVKDELMFSPLVDSGS
NDAMAGGFTAYGGADYGEATSPFGAYGEAPTSPGFGVSSPGFSPTSPTYSPTSPAYSPTS
PSYSPTSPSYSPTSPSYSPTSPSYSPTSPSYSPTSPSYSPTSPSYSPTSPSYSPTSPSYS
PTSPSYSPTSPSYSPTSPSYSPTSPSYSPTSPSYSPTSPAYSPTSPSYSPTSPSYSPTSP
SYSPTSPSYSPTSPNYSPTSPSYSPTSPGYSPGSPAYSPKQDEQKHNENENSR"""

#converts original fasta file to a string
rbp1_fasta_file = StringIO(rbp1_fasta)

#extract only the sequences out
rbp1_full_sequence = str(SeqIO.read(rbp1_fasta_file, "fasta").seq)

#extract only the trigger loop sequence
#since UniProt in 1-indexed and python's 0-indexed, we start at one less the desired index
#trigger loop spans residues 1076-1106 (per Dutagaci et al. 2023)
trigger_loop_start = 1075
trigger_loop_end = 1106
trigger_loop_sequence = rbp1_full_sequence[trigger_loop_start:trigger_loop_end]

#serve as a checkpoint for correct trigger loop sequence extraction
assert (trigger_loop_sequence[2] == 'Q' and trigger_loop_sequence[9] == 'H' and trigger_loop_sequence[5] == 'L' and trigger_loop_sequence[27] == 'E'), "Extracted trigger loop sequence is not accurate!"
print(trigger_loop_sequence, len(trigger_loop_sequence))

#use pandas to extract mutant and fitness data
df = pd.read_excel('journal.pcbi.1010999.s029.xlsx', sheet_name='Training data')

#create and save mutant sequence in DataFrame
for m in range(len(df['Mutant'])):
  mutant = df['Mutant'][m]
  mutant_position = int(mutant[1:-1])
  mutant_new_ac = mutant[-1]
  mutant_trigger_loop_sequence = trigger_loop_sequence[0:(mutant_position-1076)] + mutant_new_ac + trigger_loop_sequence[(mutant_position-1075):len(trigger_loop_sequence)]
  df.loc[m, 'Mutant Sequence'] = mutant_trigger_loop_sequence

#upload esm 2 model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t12_35M_UR50D")
model = EsmModel.from_pretrained("facebook/esm2_t12_35M_UR50D")

embedding_results = []
fitness_data = []

# run each mutant sequence through the esm 2 model
# change the embedding vector into a numpy array while storing it in a list

"""
for i in range(len(df)):
  test_sequence = df['Mutant Sequence'][i]
  inputs = tokenizer(test_sequence, return_tensors="pt")
  outputs = model(**inputs)

  embedding_output = outputs.last_hidden_state.mean(dim=1).detach().numpy()
  embedding_results.append(embedding_output)

embedding_results = np.array(embedding_results).squeeze()
"""
for i in range(len(df)):
  test_sequence = df['Mutant Sequence'][i]
  inputs = tokenizer(test_sequence, return_tensors="pt")
  outputs = model(**inputs)

  mutant = df['Mutant'][i]
  mutant_position = int(mutant[1:-1])
  token_index = (mutant_position-1076)+1

  embedding_output = outputs.last_hidden_state.detach()
  mutant_embedding = embedding_output[0, token_index].numpy()
  embedding_results.append(mutant_embedding)

embedding_results = np.array(embedding_results).squeeze()

#change fitness data into a numpy array and store it in list
#same index as the embedding vector for each mutant
fitness_data = np.array(df['Continous Phenotype from Fitness Data'])

embedding_train, embedding_test, fitness_train, fitness_test = train_test_split(
    embedding_results, fitness_data, test_size=0.2, random_state=42
)

print(f"Embedding_train shape: {embedding_train.shape}, Fitness_train shape: {fitness_train.shape}")
print(f"Embedding_test shape: {embedding_test.shape}, Fitness_test shape: {fitness_test.shape}")

#alphas_list=[0.01, 0.1, 1.0, 10.0, 100.0]
model = Ridge()
#model.fit(embedding_train, fitness_train)
score = cross_val_score(model, embedding_train, fitness_train, cv=5, scoring='r2')
print(f"Score: {score} Score mean: {score.mean()} Score STD: {score.std()}")


#print(f"R2 score: {r2_score(fitness_test, model.predict(embedding_test))}, Alpha: {model.alpha_}")







ATQMTLNTFHFAGVASKKVTSGVPRLKEILN 31


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Embedding_train shape: (66, 480), Fitness_train shape: (66,)
Embedding_test shape: (17, 480), Fitness_test shape: (17,)
Score: [-0.09942628  0.07793138  0.53466123 -0.61174138  0.12576402] Score mean: 0.005437794213190084 Score STD: 0.371945635889501


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving journal.pcbi.1010999.s029.xlsx to journal.pcbi.1010999.s029 (1).xlsx
